#  Libraries 
We will be using the following library for our functions that we need

In [35]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import KFold

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score


## Preprocessing

In [59]:
def load_data(path):
    df = pd.read_csv(path)
    feature_names = df.columns[:-1].tolist()
    X = df.drop("target", axis=1).values
    y = df["target"].values
    return X, y, feature_names

def split_data(X, y):
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
    return X_train, X_val, X_test, y_train, y_val, y_test

def normalize_data(X_train, X_val, X_test):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)
    return X_train, X_val, X_test

# SVM

Using the code below, we used the SVM baseline with no feature transformation.

In [85]:
def one_vs_rest_encoding(y, digit=1):
    return np.where(y == digit, 1, -1)

def score(X, w):
    return X.dot(w)

def svm_objective(w, X, y, lambda1=0.08):
    """Computes the Primal SVM objective function: 
    L(w) = lambda/2 * ||w||^2 + hinge loss"""
    result = lambda1/2.0 * np.dot(w,w) + np.mean(np.maximum(0, 1 - y * score(X, w)))
    return result

def pegasos(X_train, y_train, lambda1=0.08, num_iters=3):
    N = X_train.shape[0] 
    d = X_train.shape[1]
    t = 0
    # Initial weight vector w as a vector of ones
    w = np.ones((d,))

    for iter in range(num_iters):
        for i in range(N):
            t = t + 1
            # Step size and regularization scaling
            w = (1 - 1/t) * w
            # Subgradient update if margin is violated (y * score < 1)
            if y_train[i] * score(X_train[i], w) < 1:
                w = w + (1/(lambda1*t)) * y_train[i] * X_train[i]
    
        
    return w

X_raw, y_raw, feature_names = load_data("heart.csv")
y_encoded = one_vs_rest_encoding(y_raw)

X_train, X_val, X_test, y_train, y_val, y_test = split_data(X_raw, y_encoded)

X_train_scaled, X_val_scaled, X_test_scaled = normalize_data(X_train, X_val, X_test)

lambdas = [0.001, 0.01, 0.1, 1, 10, 100]
best_lambda = None
best_val_acc = -1

print(f"{'Lambda':<10} | {'Validation Accuracy':<20}")
print("-" * 35)

for l in lambdas:
    w = pegasos(X_train_scaled, y_train, lambda1=l, num_iters=15)
    
    val_scores = score(X_val_scaled, w)
    val_preds = np.where(val_scores > 0, 1, -1)
    val_acc = np.mean(val_preds == y_val)
    
    print(f"{l:<10} | {val_acc*100:18.2f}%")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_lambda = l

print("-" * 35)
print(f"Best Lambda found: {best_lambda}")

w_final = pegasos(X_train_scaled, y_train, lambda1=best_lambda, num_iters=15)
test_scores = score(X_test_scaled, w_final)
test_preds = np.where(test_scores > 0, 1, -1)

final_accuracy = np.mean(test_preds == y_test)
tp = np.sum((test_preds == 1) & (y_test == 1))
fp = np.sum((test_preds == 1) & (y_test == -1))
fn = np.sum((test_preds == -1) & (y_test == 1))

final_precision = tp / (tp + fp) if (tp + fp) > 0 else 0
final_recall = tp / (tp + fn) if (tp + fn) > 0 else 0

print(f"\n--- Final Results on Test Set (Lambda={best_lambda}) ---")
print(f"Accuracy:  {final_accuracy:.4f}")
print(f"Precision: {final_precision:.4f}")
print(f"Recall:    {final_recall:.4f}")

Lambda     | Validation Accuracy 
-----------------------------------
0.001      |              82.35%
0.01       |              83.33%
0.1        |              83.33%
1          |              82.35%
10         |              76.47%
100        |              76.47%
-----------------------------------
Best Lambda found: 0.01

--- Final Results on Test Set (Lambda=0.01) ---
Accuracy:  0.7670
Precision: 0.7458
Recall:    0.8302


# Adding Feature Transformation

One of the feature Transformation that we added is RBF Kernel. Using some of the functions earlier, we also implemented the following code for RBF Kernel:

In [ ]:
def rbf_kernel(x1, x2, gamma=0.1):
    x1 = np.atleast_2d(x1)
    x2 = np.atleast_2d(x2)
    
    # ||a - b||^2 = a^2 + b^2 - 2ab
    sq_norm1 = np.sum(x1**2, axis=1).reshape(-1, 1)
    sq_norm2 = np.sum(x2**2, axis=1)
    
    sq_dist = sq_norm1 + sq_norm2 - 2 * np.dot(x1, x2.T)
    
    K = np.exp(-gamma * sq_dist)

    return K.flatten() if K.shape[1] == 1 or K.shape[0] == 1 else K

def kernel_pegasos(X_train, y_train, lambda1=0.08, gamma=0.1, num_iters=3):
    N = X_train.shape[0]
    alpha = np.zeros(N)
    t = 0

    for _ in range(num_iters):
        for i in range(N):
            t += 1
            # score = sum(alpha_j * y_j * K(x_j, x_i))
            support_indices = np.where(alpha > 0)[0]
            
            if len(support_indices) == 0:
                current_score = 0
            else:
                kernels = rbf_kernel(X_train[support_indices], X_train[i], gamma)
                current_score = np.sum(alpha[support_indices] * y_train[support_indices] * kernels)
            
            # y_i * f(x_i) < 1
            if y_train[i] * (current_score / (lambda1 * t)) < 1:
                alpha[i] += 1
                
    return alpha, X_train, y_train

def kernel_predict(X_test, alpha, X_train, y_train, lambda1, gamma):
    N_test = X_test.shape[0]
    predictions = np.zeros(N_test)
    
    support_indices = np.where(alpha > 0)[0]
    
    for i in range(N_test):
        # f(x) = (1 / lambda * T) * sum(alpha_j * y_j * K(x_j, x_test))
        kernels = rbf_kernel(X_train[support_indices], X_test[i], gamma)
        score = np.sum(alpha[support_indices] * y_train[support_indices] * kernels)
        predictions[i] = 1 if score > 0 else -1
        
    return predictions


Kernel SVM Accuracy: 0.9220
Kernel SVM Precision: 0.8783
Kernel SVM Recall: 0.9806


In [73]:
def SVM_RBF_search(X_train, y_train, X_val, y_val, X_test, y_test, fixed_gamma=0.1):
    lambdas = [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0]
    best_l_rbf = None
    best_v_acc_rbf = -1

    print(f"{'Lambda':<10} | {'Val Accuracy':<15}")
    print("-" * 30)

    for l in lambdas:
        alpha, X_sv, y_sv = kernel_pegasos(X_train, y_train, lambda1=l, gamma=fixed_gamma, num_iters=5)
        
        preds = kernel_predict(X_val, alpha, X_sv, y_sv, l, fixed_gamma)
        val_acc = np.mean(preds == y_val)
        
        print(f"{l:<10} | {val_acc*100:12.2f}%")
        
        if val_acc > best_v_acc_rbf:
            best_v_acc_rbf = val_acc
            best_l_rbf = l

    print("-" * 30)
    print(f"Best RBF Lambda for Validation: {best_l_rbf}")

    alpha_f, X_f, y_f = kernel_pegasos(X_train, y_train, lambda1=best_l_rbf, gamma=fixed_gamma, num_iters=5)
    test_preds = kernel_predict(X_test, alpha_f, X_f, y_f, best_l_rbf, fixed_gamma)

    final_acc = np.mean(test_preds == y_test)
    tp = np.sum((test_preds == 1) & (y_test == 1))
    fp = np.sum((test_preds == 1) & (y_test == -1))
    fn = np.sum((test_preds == -1) & (y_test == 1))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    print(f"\n--- Final RBF Results on Test Set (Lambda={best_l_rbf}) ---")
    print(f"Accuracy:  {final_acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")


SVM_RBF_search(X_train_scaled, y_train, X_val_scaled, y_val, X_test_scaled, y_test, fixed_gamma=0.1)

Lambda     | Val Accuracy   
------------------------------
0.0001     |        99.02%
0.001      |        90.20%
0.01       |        85.29%
0.1        |        77.45%
1.0        |        77.45%
10.0       |        77.45%
------------------------------
Best RBF Lambda for Validation: 0.0001

--- Final RBF Results on Test Set (Lambda=0.0001) ---
Accuracy:  0.9515
Precision: 0.9444
Recall:    0.9623


# Table for SVM RBF Kernel

In order to produce the table with the dadtaset, I've added the following code to assist me with each lambda and gamma value for RBF Kernel:

# Second and Third Feature Transformation for SVM
The second and third freature transformation we did was polynomial kernel. We did the 2nd degree, 3rd degree, and for the fun of it, we did the 4th degree as well. We switched to the polynomial kernel because it felt on the rbf kernel, it was overfitting really early on. So we tested it in lower degree and it seems like the 2nd degree is a lot better compared to the 3rd and 4th degree where it started to show sign of overfitting

In [77]:
def sklearn_poly_svm_search(X_train, y_train, X_test, y_test, degree1 = 2):

    c_values = [0.001, 0.01, 0.1, 1.0,10, 100.0]
    best_c_poly = None
    best_v_acc_poly = -1

    print(f"{'C (Reg)':<10} | {'Val Accuracy':<15}")
    print("-" * 30)

    for c in c_values:
        clf = SVC(kernel='poly', degree=degree1, C=c, gamma='scale', coef0=1)
        clf.fit(X_train_scaled, y_train)
        
        val_acc = clf.score(X_val_scaled, y_val)
        
        print(f"{c:<10} | {val_acc*100:12.2f}%")
        
        if val_acc > best_v_acc_poly:
            best_v_acc_poly = val_acc
            best_c_poly = c

    print("-" * 30)
    print(f"Best C for Polynomial: {best_c_poly}")

    clf_final = SVC(kernel='poly', degree=degree1, C=1, gamma='scale', coef0=1)
    clf_final.fit(X_train_scaled, y_train)

    test_preds = clf_final.predict(X_test_scaled)

    final_acc = accuracy_score(y_test, test_preds)
    final_prec = precision_score(y_test, test_preds, zero_division=0)
    final_rec = recall_score(y_test, test_preds, zero_division=0)

    print(f"\n Final Polynomial Results on Test Set (C={1})")
    print(f"Accuracy:  {final_acc:.4f}")
    print(f"Precision: {final_prec:.4f}")
    print(f"Recall:    {final_rec:.4f}")

# To run this in your notebook:
print("Polynomial SVM with degree 2:")
sklearn_poly_svm_search(X_train_scaled, y_train, X_test_scaled, y_test, 2)
print("\nPolynomial SVM with degree 3:")
sklearn_poly_svm_search(X_train_scaled, y_train, X_test_scaled, y_test, 3)
print("\nPolynomial SVM with degree 4:")
sklearn_poly_svm_search(X_train_scaled, y_train, X_test_scaled, y_test, 4)



Polynomial SVM with degree 2:
C (Reg)    | Val Accuracy   
------------------------------
0.001      |        49.02%
0.01       |        80.39%
0.1        |        83.33%
1.0        |        85.29%
10         |        91.18%
100.0      |        98.04%
------------------------------
Best C for Polynomial: 100.0

 Final Polynomial Results on Test Set (C=1)
Accuracy:  0.8738
Precision: 0.8448
Recall:    0.9245

Polynomial SVM with degree 3:
C (Reg)    | Val Accuracy   
------------------------------
0.001      |        66.67%
0.01       |        82.35%
0.1        |        87.25%
1.0        |        93.14%
10         |        99.02%
100.0      |        99.02%
------------------------------
Best C for Polynomial: 10

 Final Polynomial Results on Test Set (C=1)
Accuracy:  0.9320
Precision: 0.9107
Recall:    0.9623

Polynomial SVM with degree 4:
C (Reg)    | Val Accuracy   
------------------------------
0.001      |        75.49%
0.01       |        81.37%
0.1        |        91.18%
1.0     